# WEEK 8｜Pandas 缺失值与多级索引

source 聚焦缺失数据处理、多级索引（MultiIndex）、索引层级上的筛选与重排，以及 stack/unstack。

## 1. 缺失值处理

对表格先统计各列缺失值，再根据任务决定删除或填补。source 练习依次要求：统计缺失数量、移除指定列、删除缺失值过多的行、用前值填充、按列均值填充并保留原 DataFrame。每个步骤保存为新对象，便于比较处理结果。

In [ ]:
import pandas as pd

df = pd.DataFrame({
    "sales": [10.0, None, 30.0],
    "ord_date": ["Mon", "Tue", "Wed"],
    "income": [100.0, 120.0, None],
})
print(df.isna().sum())

df1 = df.drop(columns=["ord_date"])
df2 = df1.dropna(thresh=df1.shape[1] - 1)
df3 = df2.ffill()
df4 = df2.fillna(df2.mean().round(1))
print(df2, df3, df4, sep="\n\n")

`isna().sum()` 按列统计缺失值。`drop(columns=...)` 删除列；`dropna` 删除满足缺失条件的行；`ffill` 用前一项填充；`fillna` 用给定值填充。source 的练习强调从 df2 分别创建 df3、df4，不覆盖 df2。

## 2. MultiIndex 是什么

MultiIndex 在一个轴上使用多个索引层级。例如人口数据可先按州、再按年份索引；二维 DataFrame 仍能表达更有层次的记录。给层级命名能让索引含义更清楚。

In [ ]:
index = pd.MultiIndex.from_product(
    [["California", "Texas"], [2000, 2010]],
    names=["State", "Year"],
)
population = pd.Series([34, 25, 37, 29], index=index, name="Population")
print(population)

## 3. 选择与切片

按完整元组可以定位一条多级索引记录；按外层标签可以选择该组全部记录。也可对 Series 做布尔筛选。若按标签范围切片，source 提醒先对索引排序。若要按内层级别查询，可用 `xs()` 指定层级。

In [ ]:
print(population.loc[("California", 2010)])
print(population.loc["California"])
print(population[population > 30])
print(population.xs(2010, level="Year"))

DataFrame 可在行或列使用 MultiIndex。用 `loc` 选择标签，用 `iloc` 按整数位置选择；对多级列标签也可传入元组。切换层级顺序后，索引的显示与查找层次随之变化。

In [ ]:
health = pd.DataFrame(
    [[70, 72], [75, 77]],
    index=[2013, 2014],
    columns=pd.MultiIndex.from_product(
        [["Bob", "Sue"], ["HR"]],
        names=["Person", "Measure"],
    ),
)
print(health.iloc[:2, :])
print(health.loc[:, ("Bob", "HR")])

## 4. stack 与 unstack

`stack()` 将列层级移到行索引中，通常使表格变长；`unstack()` 将内层行索引移回列，通常使表格变宽。它们用于在宽表与带多级索引的长表之间重排数据。

In [ ]:
wide = population.unstack("Year")
print(wide)
print(wide.stack())

## 5. 设置和重置索引

`reset_index()` 把原索引层级转成普通列；设置索引则可把列作为索引。source 练习用此过程把 MultiIndex Series 转成 DataFrame，让层级信息回到列中。

In [ ]:
as_columns = population.reset_index()
print(as_columns)
back_to_index = as_columns.set_index(["State", "Year"])["Population"]
print(back_to_index)

## 6. 重排索引层级与分组聚合

MultiIndex 可以调整层级顺序；若要按标签范围切片，先排序索引。对某一索引层级分组后，可以在该层级上汇总数据。

In [ ]:
reordered = population.swaplevel("State", "Year").sort_index()
print(reordered)
print(population.groupby(level="Year").mean())

source 的 Data Aggregation 练习还要求交换 `visit` 与 `year` 两个索引层级，再选取 Bob 和 Sue 的 Heart Rate，并限定为 2013 与 2014 年各自的第一次访问。这是把层级重排与多级标签选择连起来的例子。

## 本周回顾

缺失值处理包括统计、删行/删列、前向填充和均值填充；多级索引允许多个标签层级共同定位记录。用层级标签、布尔条件和 `xs` 选择数据；用 stack/unstack 重排行列层级；用 reset_index 在索引和列之间转换。